# FinLedger - Hybrid Bank Transaction Classifier (First-Semester ML Project)

This is the **main Machine Learning notebook** for my first-semester project. Run each cell from top to bottom (`Shift + Enter`).

**Goal:** Give the system a bank transaction description (like `UPI/P2M/738291048291/ZOMATO` or `POS DEBIT 192837465019 KRISHNA SUPERMARKET`) and let it predict the right spending category (`Food & Drinks`, `Groceries`, etc.).

**Why a Two-Stage Hybrid Architecture?**
In my initial experiments, a standard Word TF-IDF + Logistic Regression model scored **99%+ accuracy** on a random train/test split because the exact same merchant names appeared in both training and testing. However, when tested on **unseen merchants** (for example, training on `SWIGGY` and testing on `ZOMATO`, which share zero common letters), accuracy collapsed!

To solve this in a beginner-friendly, transparent way, this notebook uses **Method 2 (Two-Stage Hybrid Pipeline)**:
1. **Stage 1 (Known-Merchant Prior Dictionary):** Checks if the transaction contains a well-known Indian brand or app (`ZOMATO`, `SWIGGY`, `FLIPKART`, `UBER`, `ZERODHA`, etc.). If found, it assigns the category directly with 100% confidence.
2. **Stage 2 (Cleaned Merchant Text + Word & Character N-Gram TF-IDF + Logistic Regression):** If the merchant is not in our dictionary (such as a local shop like `SHARMA SWEETS` or `MEDPLUS PHARMACY`), we first strip payment-channel noise (`UPI/P2M`, `POS DEBIT`, 12-digit reference IDs) so the ML model never memorizes banking prefixes, and then extract both **word phrases** and **3-to-5 letter character subwords (`char_wb`)** to learn morphological roots (`MART`, `CAFE`, `PHARM`, `CLINIC`, `PETROL`, `REST`).

## Cell 1 - Import our tools

- **Pandas & NumPy** handle tables and math operations.
- **Scikit-learn** builds our TF-IDF vectorizers, Logistic Regression classifier, evaluation metrics, and Isolation Forest.
- **Joblib** saves our dictionary and trained ML model together into one file (`model.joblib`) for our Streamlit web app.

In [2]:
from pathlib import Path
import re
import joblib
import numpy as np
import pandas as pd
import requests
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.ensemble import IsolationForest

print("Libraries imported successfully!")

Libraries imported successfully!


## Cell 2 - Load and combine the training datasets

The public Hugging Face dataset (`data/bank_dataset.csv`) is great for national brands, but it only has **50 distinct merchant templates** (all single-word names like `ZOMATO`, `OLACABS`, `WATERSUPPLYDEPT`) and lacks local Indian shop descriptions (`SHARMA SWEETS`, `KRISHNA SUPERMARKET`, `MEDPLUS PHARMACY`).

So if `data/bank_dataset.csv` is available, we **combine it** with our local merchant dataset (`data/demo_training.csv`). That way our model learns both national brands and local descriptor-based shops across 220+ merchant groups!

In [17]:
real_file = Path('data/bank_dataset.csv')
demo_file = Path('data/demo_training.csv')
url = 'https://huggingface.co/datasets/karthiksagarn/bank-statement-categorization/resolve/main/bank_dataset.csv'

if not real_file.exists():
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        real_file.write_bytes(response.content)
        print('Downloaded the public Hugging Face dataset.')
    except Exception:
        print('Offline mode: using local dataset data/demo_training.csv.')

dfs = []
if real_file.exists():
    dfs.append(pd.read_csv(real_file))
if demo_file.exists():
    dfs.append(pd.read_csv(demo_file))

data = pd.concat(dfs, ignore_index=True)
# is_demo = not real_file.exists()
print(f'Loaded {len(dfs)} dataset source(s) | Total raw rows: {len(data)}')
data.head()

Loaded 2 dataset source(s) | Total raw rows: 13012


,description,category
0,UPI/P2M/695829655510/COURSERA,Education
1,UPI/P2M/235190313324/MAKEMYTRIP,Travel & Transport
2,UPI/P2M/905651248723/JIOMART,Groceries
3,UPI/P2M/720580055542/BLINKIT,Groceries
4,UPI/P2M/516707692505/GIFT,Miscellaneous


## Cell 3 - Clean data, strip banking boilerplate, and understand X and y

When I inspected the learned weights of my earlier model on the public dataset, I noticed two critical bugs:
1. Features like `W:p2m zomato` and `W:debit localgym` were at the top of the weight table! Because `UPI/P2M` and `POS DEBIT` were left inside the training text, any unknown vendor starting with `UPI/P2M/...` (like `UPI/P2M/818192/X99Z UNKNOWN VENDOR`) got a **falsely high 68.2% confidence score** just from the words `UPI` and `P2M`!
2. To prevent the ML model from getting tricked by payment channel prefixes or 12-digit reference numbers, we define `clean_merchant_text()` to strip `UPI`, `P2M`, `POS`, `DEBIT`, `IMPS`, and long reference IDs before Stage 2 ML sees the text.

In [23]:
def clean_merchant_text(description):
    """Strips payment channel prefixes (UPI, P2M, POS DEBIT) and long reference IDs so ML only sees the merchant words."""
    text = str(description).upper()
    text = re.sub(r'[/\-_]+', ' ', text)
    text = re.sub(r'\b\d{4,}\b', ' ', text)
    text = re.sub(r'\b(UPI|P2M|P2A|POS|DEBIT|IMPS|ACH|REF|ONLINE|ORDER|PAYMENT|TXN|NO)\b', ' ', text)
    return ' '.join(text.split()).strip() or 'UNKNOWN'

data = data[['description', 'category']].dropna().drop_duplicates().copy()
data['description'] = data['description'].astype(str).str.strip()
data['category'] = data['category'].astype(str).str.strip()
data = data[(data['description'] != '') & (data['category'] != '')].reset_index(drop=True)

X = data['description']
y = data['category']

print('Rows after removing duplicates:', len(data))
print('Number of categories:', y.nunique())
print('Sample cleaned merchant text:')
for raw_tr in ['UPI/P2M/695829655510/COURSERA', 'POS DEBIT 583920 MEDPLUS PHARMACY', 'UPI/P2M/818192/X99Z UNKNOWN VENDOR']:
    print(f"  {raw_tr}  -->  '{clean_merchant_text(raw_tr)}'")

Rows after removing duplicates: 11367
Number of categories: 12
Sample cleaned merchant text:
  UPI/P2M/695829655510/COURSERA  -->  'COURSERA'
  POS DEBIT 583920 MEDPLUS PHARMACY  -->  'MEDPLUS PHARMACY'
  UPI/P2M/818192/X99Z UNKNOWN VENDOR  -->  'X99Z UNKNOWN VENDOR'


## Cell 4 - Stage 1: Build the Prior Dictionary for Popular Brands

Why do we need a dictionary before running Machine Learning?
From first principles, two famous brands like `ZOMATO` and `SWIGGY` (or `PVR` and `INOX`) share almost no common letters. No character n-gram or word TF-IDF model can guess that `ZOMATO` means food if it only saw `SWIGGY` during training.

In **Method 2 (Two-Stage Hybrid)**:
- **Stage 1 (`rule_based_lookup`)** checks our dictionary of popular food delivery apps, shopping apps, cinema chains, and utility providers first.
- Notice we also check `tok.startswith(kw)` for brand roots of 4+ letters so concatenated names in the public dataset like `OLACABS` (`OLA`), `PVRCINEMAS` (`PVR`), and `RELIANCEDIGITAL` are caught cleanly!

In [27]:
KNOWN_MERCHANT_DICT = {
    'Food & Drinks': [
        'SWIGGY', 'ZOMATO', 'DOMINOS', 'KFC', 'MCDONALDS', 'STARBUCKS', 'BURGERKING', 'EATCLUB', 'PIZZAHUT', 'BEERCAFE', 'THEBEERCAFE'
    ],
    'Groceries': [
        'BLINKIT', 'ZEPTO', 'BIGBASKET', 'JIOMART', 'DMART', 'INSTAMART', 'RELIANCEFRESH', 'NATUREBASKET'
    ],
    'Shopping': [
        'FLIPKART', 'AMAZON', 'MYNTRA', 'AJIO', 'MEESHO', 'NYKAA', 'DECATHLON', 'CROMA', 'TATACLIQ', 'RELIANCEDIGITAL'
    ],
    'Travel & Transport': [
        'UBER', 'OLA', 'OLACABS', 'RAPIDO', 'IRCTC', 'MAKEMYTRIP', 'REDBUS', 'INDIANRAILWAYS', 'INDIGO', 'GOIBIBO', 'YATRA'
    ],
    'Bills & Utilities': [
        'ELECTRICITYBOARD', 'AIRTEL', 'JIOFIBER', 'VODAFONE', 'TATAPOWER', 'BESCOM', 'ACTFIBERNET', 'BSNL', 'WATERSUPPLYDEPT'
    ],
    'Education': [
        'COURSERA', 'UDEMY', 'UNACADEMY', 'BYJUS', 'UNIVERSITYFEES', 'COLLEGEFEES', 'EDX'
    ],
    'Entertainment': [
        'NETFLIX', 'SPOTIFY', 'BOOKMYSHOW', 'PVR', 'PVRCINEMAS', 'INOX', 'HOTSTAR', 'PRIMEVIDEO', 'SONYLIV'
    ],
    'Health & Fitness': [
        'APOLLOPHARMACY', 'PHARMEASY', 'NETMEDS', '1MG', 'LOCALGYM', 'CULTFIT', 'MYPROTEIN', 'PRACTO'
    ],
    'Investments': [
        'ZERODHA', 'GROWW', 'UPSTOX', 'ANGELONE', 'LIC INDIA', 'MUTUALFUND'
    ]
}

def rule_based_lookup(description, merchant_dict=KNOWN_MERCHANT_DICT):
    text = str(description).upper()
    clean_tokens = set(re.sub(r'[^A-Z0-9]+', ' ', text).split())
    for category, keywords in merchant_dict.items():
        for kw in keywords:
            if ' ' in kw:
                if kw in text:
                    return category
            else:
                if kw in clean_tokens or (len(kw) >= 4 and any(token.startswith(kw) for token in clean_tokens)):
                    return category
    return None

print("Testing Dictionary Lookup:")
print("1. UPI/P2M/738291/ZOMATO ->", rule_based_lookup("UPI/P2M/738291/ZOMATO"))
print("2. POS/DEBIT/612484/OLACABS ->", rule_based_lookup("POS/DEBIT/612484/OLACABS"))
print("3. POS/DEBIT/192837/KRISHNA/SUPERMARKET ->", rule_based_lookup("POS/DEBIT/192837/KRISHNA/SUPERMARKET"))

Testing Dictionary Lookup:
1. UPI/P2M/738291/ZOMATO -> Food & Drinks
2. POS/DEBIT/612484/OLACABS -> Travel & Transport
3. POS/DEBIT/192837/KRISHNA/SUPERMARKET -> None


## Cell 5 - Stage 2: Build the Word + Character N-Gram ML Model

When Stage 1 returns `None` (like `KRISHNA SUPERMARKET` or `MEDPLUS PHARMACY`), **Stage 2 (Machine Learning)** takes over on the **cleaned merchant text**!

We use `FeatureUnion` to combine two TF-IDF feature extractors:
1. **`word_tfidf`:** Captures whole words and 2-word phrases (`SUPERMARKET`, `SALARY CREDIT`, `PETROL PUMP`).
2. **`char_tfidf`:** Breaks words into 3-to-5 letter subwords (`analyzer='char_wb'`). Even if a new local shop is concatenated (like `WATERSUPPLYDEPT`, `THEBEERCAFE`, or `FRESHVEGGIEMART`), character n-grams recognize roots like `WATER`, `CAFE`, `MART`, `PHARM`, and `GYM`.
3. **`classifier`:** **Logistic Regression** learns weights for all these word and character features to predict the category.

In [28]:
def build_ml_pipeline():
    features = FeatureUnion([
        ('word_tfidf', TfidfVectorizer(
            ngram_range=(1, 2),
            max_features=10000,
            token_pattern=r'(?u)\b[a-zA-Z][a-zA-Z0-9]+\b',
            sublinear_tf=True
        )),
        ('char_tfidf', TfidfVectorizer(
            analyzer='char_wb',
            ngram_range=(3, 5),
            max_features=15000,
            sublinear_tf=True
        ))
    ])
    
    model = Pipeline([
        ('features', features),
        ('classifier', LogisticRegression(
            max_iter=1500,
            class_weight='balanced',
            random_state=42
        ))
    ])
    return model

ml_model = build_ml_pipeline()
print(ml_model)

Pipeline(steps=[('features',
                 FeatureUnion(transformer_list=[('word_tfidf',
                                                 TfidfVectorizer(max_features=10000,
                                                                 ngram_range=(1,
                                                                              2),
                                                                 sublinear_tf=True,
                                                                 token_pattern='(?u)\\b[a-zA-Z][a-zA-Z0-9]+\\b')),
                                                ('char_tfidf',
                                                 TfidfVectorizer(analyzer='char_wb',
                                                                 max_features=15000,
                                                                 ngram_range=(3,
                                                                              5),
                                                                 

## Cell 6 - Connect Stage 1 and Stage 2 into Our Hybrid Predictor

Here is how our complete **Method 2 (Rule First, ML Second)** function works on any list of transactions:
1. First, it checks `rule_based_lookup(text)`. If the merchant is in `KNOWN_MERCHANT_DICT`, it assigns that category with `confidence = 1.00` and marks the source as `'Stage 1: Dictionary'`.
2. If not found in the dictionary, it cleans the text with `clean_merchant_text(text)` and uses the Stage 2 ML model (`predict_proba`) to get the most likely category and its probability score.
3. **Confidence Threshold:** Because we stripped `UPI/P2M` and `POS DEBIT`, a mysterious vendor like `UPI/P2M/818192/X99Z UNKNOWN VENDOR` now gets a flat, low probability (~0.11–0.15) instead of 0.68, reliably triggering `needs_review = True` when confidence is below `0.45`!

In [29]:
def hybrid_predict_df(descriptions, trained_ml_model, merchant_dict=KNOWN_MERCHANT_DICT, conf_threshold=0.45):
    desc_list = list(descriptions)
    cleaned_list = [clean_merchant_text(d) for d in desc_list]
    classes = trained_ml_model.classes_
    
    ml_probs = trained_ml_model.predict_proba(cleaned_list)
    ml_best_idx = np.argmax(ml_probs, axis=1)
    ml_best_cat = classes[ml_best_idx]
    ml_best_conf = np.max(ml_probs, axis=1) # argmax() returns the position and max() returns the value.
    
    rows = []
    for i, text in enumerate(desc_list):
        rule_cat = rule_based_lookup(text, merchant_dict)
        if rule_cat is not None:
            rows.append({
                'description': text,
                'predicted': rule_cat,
                'confidence': 1.00,
                'source': 'Stage 1: Dictionary',
                'needs_review': False
            })
        else:
            conf = float(ml_best_conf[i])
            rows.append({
                'description': text,
                'predicted': str(ml_best_cat[i]),
                'confidence': round(conf, 3),
                'source': 'Stage 2: ML (Word+Char TF-IDF)',
                'needs_review': bool(conf < conf_threshold)
            })
    return pd.DataFrame(rows)

print("Hybrid predictor function ready!")

Hybrid predictor function ready!


## Cell 7 - Split data (80/20), train Stage 2 ML on cleaned text, and evaluate

We split our dataset into **80% training** and **20% testing** (`stratify=y` keeps category proportions balanced), fit the Stage 2 ML model on `X_train.apply(clean_merchant_text)`, and evaluate the full Two-Stage Hybrid pipeline on `X_test`.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

ml_model.fit(X_train.apply(clean_merchant_text), y_train)
hybrid_out = hybrid_predict_df(X_test, ml_model)
predictions = hybrid_out['predicted'].values

accuracy = accuracy_score(y_test, predictions)
macro_f1 = f1_score(y_test, predictions, average='macro', zero_division=0)

print(f'Training rows: {len(X_train)} | Testing rows: {len(X_test)}')
print(f'Test accuracy: {accuracy:.3f}')
print(f'Macro F1:      {macro_f1:.3f}')
print('\nHow many test transactions were handled by each stage?')
print(hybrid_out['source'].value_counts())
print('\nDetailed report:')
print(classification_report(y_test, predictions, zero_division=0))

Training rows: 9093 | Testing rows: 2274
Test accuracy: 0.995
Macro F1:      0.992

How many test transactions were handled by each stage?
source
Stage 1: Dictionary               1163
Stage 2: ML (Word+Char TF-IDF)    1111
Name: count, dtype: int64

Detailed report:
                    precision    recall  f1-score   support

 Bills & Utilities       1.00      1.00      1.00       140
         Education       1.00      1.00      1.00       215
     Entertainment       1.00      1.00      1.00       220
     Food & Drinks       1.00      1.00      1.00       216
         Groceries       1.00      1.00      1.00       230
  Health & Fitness       1.00      1.00      1.00       194
            Income       1.00      1.00      1.00       188
       Investments       1.00      1.00      1.00       186
     Miscellaneous       1.00      0.88      0.94       100
          Shopping       1.00      1.00      1.00       189
Travel & Transport       1.00      1.00      1.00       223
       With

## Cell 8 - Inspect predictions and check for errors

Below we view the first 12 test predictions showing whether **Stage 1 (Dictionary)** or **Stage 2 (ML)** handled each row, along with the confidence score. We also explicitly filter for wrong predictions (`actual != predicted`).

In [ ]:
results = hybrid_out.copy()
results.insert(1, 'actual', y_test.values)

errors = results[results['actual'] != results['predicted']]
print(f"Misclassified rows on random test split: {len(errors)} out of {len(results)}")
if len(errors) > 0:
    print("Sample misclassified rows on random split:")
    print(errors.head(12).to_string())
results.head(12)

Misclassified rows on random test split: 12 out of 2274
Sample misclassified rows on random split:
                                description         actual    predicted  confidence                          source  needs_review
50    POS DEBIT 393140292427 CASHWITHDRAWAL  Miscellaneous  Withdrawals       0.882  Stage 2: ML (Word+Char TF-IDF)         False
341   POS DEBIT 415139312652 CASHWITHDRAWAL  Miscellaneous  Withdrawals       0.882  Stage 2: ML (Word+Char TF-IDF)         False
356   POS DEBIT 396084435401 CASHWITHDRAWAL  Miscellaneous  Withdrawals       0.882  Stage 2: ML (Word+Char TF-IDF)         False
543   POS DEBIT 526902824658 CASHWITHDRAWAL  Miscellaneous  Withdrawals       0.882  Stage 2: ML (Word+Char TF-IDF)         False
924   POS DEBIT 997124796069 CASHWITHDRAWAL  Miscellaneous  Withdrawals       0.882  Stage 2: ML (Word+Char TF-IDF)         False
969   POS DEBIT 674655433998 CASHWITHDRAWAL  Miscellaneous  Withdrawals       0.882  Stage 2: ML (Word+Char TF-IDF)      

,description,actual,predicted,confidence,source,needs_review
0,POS DEBIT 596961148076 ANGELONE,Investments,Investments,1.000,Stage 1: Dictionary,False
1,UPI/P2M/251669128846/COURSERA,Education,Education,1.000,Stage 1: Dictionary,False
2,ATM WDL/678376206718/CASH WITHDRAWAL AT POS,Withdrawals,Withdrawals,0.986,Stage 2: ML (Word+Char TF-IDF),False
3,UPI/P2M/357899256307/JIOMART,Groceries,Groceries,1.000,Stage 1: Dictionary,False
4,UPI/P2M/928767771563/MYPROTEIN,Health & Fitness,Health & Fitness,1.000,Stage 1: Dictionary,False
5,POS DEBIT 239670706919 LOCALGYM,Health & Fitness,Health & Fitness,1.000,Stage 1: Dictionary,False
6,IMPS/280278023353/ROYAL CAFE AND BAKERY,Food & Drinks,Food & Drinks,0.937,Stage 2: ML (Word+Char TF-IDF),False
7,UPI/P2M/496312025663/JIOMART,Groceries,Groceries,1.000,Stage 1: Dictionary,False
8,UPI/P2M/334938298653/UDEMY,Education,Education,1.000,Stage 1: Dictionary,False
9,UPI/P2M/354753193899/AMAZON,Shopping,Shopping,1.000,Stage 1: Dictionary,False


## Cell 9 - The Unseen-Merchant Experiment (Proving Our Upgrade Works!)

In this experiment, we group transactions by merchant name using `GroupShuffleSplit` so that **every merchant in the 20% test set is 100% unseen during training**.
We compare three models side-by-side on this tough test:
1. **Old Baseline (Raw Text + Word-only TF-IDF):** Struggles when merchant names change and gets distracted by raw `UPI/P2M` prefixes.
2. **Stage 2 ML Only (Cleaned Text + Word & Character N-Grams):** Improves dramatically because stripping payment noise and extracting character subwords (`MART`, `CAFE`, `PHARM`, `GYM`, `WATER`) matches roots across different shops.
3. **Full Two-Stage Hybrid (Dictionary + Cleaned Word/Char ML):** Combines our dictionary for famous brands (`ZOMATO`, `SWIGGY`, `FLIPKART`, `OLACABS`) with character/word TF-IDF for local shops!

In [10]:
groups = data['description'].apply(clean_merchant_text)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_ids, test_ids = next(splitter.split(X, y, groups=groups))

X_grp_train, y_grp_train = X.iloc[train_ids], y.iloc[train_ids]
X_grp_test, y_grp_test = X.iloc[test_ids], y.iloc[test_ids]

# 1. Old Baseline (Raw text, Word-only TF-IDF)
old_baseline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=15000,
                              token_pattern=r'(?u)\b[a-zA-Z][a-zA-Z]+\b')),
    ('classifier', LogisticRegression(max_iter=1500, class_weight='balanced', random_state=42))
])
old_baseline.fit(X_grp_train, y_grp_train)
pred_old = old_baseline.predict(X_grp_test)

# 2. Stage 2 ML Only (Cleaned text + Word & Character N-grams)
stage2_only = build_ml_pipeline()
stage2_only.fit(X_grp_train.apply(clean_merchant_text), y_grp_train)
pred_stage2 = stage2_only.predict(X_grp_test.apply(clean_merchant_text))

# 3. Full Two-Stage Hybrid (Dictionary + Cleaned Word/Char ML)
hybrid_grp_df = hybrid_predict_df(X_grp_test, stage2_only)
pred_hybrid = hybrid_grp_df['predicted'].values

unseen_acc_old = round(accuracy_score(y_grp_test, pred_old), 3)
unseen_f1_old = round(f1_score(y_grp_test, pred_old, average='macro', zero_division=0), 3)
unseen_acc_s2 = round(accuracy_score(y_grp_test, pred_stage2), 3)
unseen_f1_s2 = round(f1_score(y_grp_test, pred_stage2, average='macro', zero_division=0), 3)
unseen_acc_hyb = round(accuracy_score(y_grp_test, pred_hybrid), 3)
unseen_f1_hyb = round(f1_score(y_grp_test, pred_hybrid, average='macro', zero_division=0), 3)

print(f"Distinct merchant groups: {groups.nunique()} | Unseen test rows: {len(X_grp_test)}")
print(f"1. Old Word-Only TF-IDF (Raw Text)         -> Accuracy: {unseen_acc_old:.3f} | Macro F1: {unseen_f1_old:.3f}")
print(f"2. Cleaned Word + Char N-Gram ML Only      -> Accuracy: {unseen_acc_s2:.3f} | Macro F1: {unseen_f1_s2:.3f}")
print(f"3. Full Two-Stage Hybrid (Dict + Clean ML) -> Accuracy: {unseen_acc_hyb:.3f} | Macro F1: {unseen_f1_hyb:.3f}")

# Inspect genuine errors on unseen merchants to understand model weaknesses
grp_results = hybrid_grp_df.copy()
grp_results.insert(1, 'actual', y_grp_test.values)
grp_errors = grp_results[grp_results['actual'] != grp_results['predicted']]
print(f"\nUnseen-merchant errors ({len(grp_errors)} out of {len(grp_results)} rows) - sample:")
grp_errors.head(10)

Distinct merchant groups: 240 | Unseen test rows: 2365
1. Old Word-Only TF-IDF (Raw Text)         -> Accuracy: 0.270 | Macro F1: 0.283
2. Cleaned Word + Char N-Gram ML Only      -> Accuracy: 0.649 | Macro F1: 0.653
3. Full Two-Stage Hybrid (Dict + Clean ML) -> Accuracy: 0.958 | Macro F1: 0.890

Unseen-merchant errors (100 out of 2365 rows) - sample:


,description,actual,predicted,confidence,source,needs_review
1248,POS DEBIT 880134027900 OXFORDBOOKSTATIONERY,Education,Travel & Transport,0.183,Stage 2: ML (Word+Char TF-IDF),True
1256,IMPS/193659379313/OXFORDBOOKSTATIONERY,Education,Travel & Transport,0.183,Stage 2: ML (Word+Char TF-IDF),True
1258,POS DEBIT 685585399705 CAMPUSBOOKSTORE,Education,Groceries,0.241,Stage 2: ML (Word+Char TF-IDF),True
1260,UPI/P2M/829712420398/OXFORDBOOKSTATIONERY,Education,Travel & Transport,0.183,Stage 2: ML (Word+Char TF-IDF),True
1281,IMPS/604808119862/CAMPUSBOOKSTORE,Education,Groceries,0.241,Stage 2: ML (Word+Char TF-IDF),True
1289,POS DEBIT 903782498625 TRENDS APPAREL STORE,Shopping,Groceries,0.387,Stage 2: ML (Word+Char TF-IDF),True
1307,POS DEBIT 760201064209 TRENDS APPAREL STORE,Shopping,Groceries,0.387,Stage 2: ML (Word+Char TF-IDF),True
1353,IMPS/561233138953/OXFORD BOOK STORE AND STATIO...,Education,Groceries,0.433,Stage 2: ML (Word+Char TF-IDF),True
1357,POS DEBIT 336172194569 OXFORD BOOK STORE AND S...,Education,Groceries,0.433,Stage 2: ML (Word+Char TF-IDF),True
1395,POS DEBIT 984256423201 TRENDS APPAREL STORE,Shopping,Groceries,0.387,Stage 2: ML (Word+Char TF-IDF),True


## Cell 10 - What did Logistic Regression actually learn? (Feature Weights)

Because we stripped `UPI/P2M` and `POS DEBIT` using `clean_merchant_text()`, look at `classifier.coef_` now: every single top feature is a genuine merchant word (`W:`) or character subword (`C:`), with zero payment-channel noise!

In [ ]:
feature_names = ml_model.named_steps['features'].get_feature_names_out()
coefs = ml_model.named_steps['classifier'].coef_
classes = ml_model.named_steps['classifier'].classes_

weight_summary = []
for idx, cat in enumerate(classes):
    top_indices = np.argsort(coefs[idx])[-5:][::-1]
    tokens = [feature_names[i].replace('word_tfidf__', 'W:').replace('char_tfidf__', 'C:') for i in top_indices]
    weight_summary.append({'Category': cat, 'Top 5 Learned Features (W=Word, C=Char N-Gram)': ', '.join(tokens)})

pd.set_option('display.max_colwidth', None) #added
pd.DataFrame(weight_summary)

,Category,"Top 5 Learned Features (W=Word, C=Char N-Gram)"
0,Bills & Utilities,"W:airtel, W:bsnl, W:watersupplydept, W:tatapow..."
1,Education,"W:coursera, W:universityfees, W:udemy, W:byjus..."
2,Entertainment,"W:bookmyshow, W:primevideo, W:netflix, W:pvrci..."
3,Food & Drinks,"W:starbucks, W:zomato, W:swiggy, W:mcdonalds, ..."
4,Groceries,"W:zepto, W:bigbasket, W:blinkit, C:mar, W:jiomart"
5,Health & Fitness,"W:myprotein, W:localgym, W:cultfit, W:apolloph..."
6,Income,"W:cr, W:neft, W:neft cr, C: cr, C:cr"
7,Investments,"W:zerodha, W:groww, W:upstox, W:angelone, W:mu..."
8,Miscellaneous,"W:gift, W:miscpurchase, W:bankcharges, W:cashw..."
9,Shopping,"W:flipkart, W:myntra, W:amazon, W:zara, W:reli..."


## Cell 11 - Train final model on ALL labeled rows and save `model.joblib`

Now that we have measured our test accuracy honestly, we retrain the Stage 2 ML model on 100% of the cleaned merchant descriptions and save both **Stage 1 (`merchant_dict`)** and **Stage 2 (`model`)** inside `model.joblib` for our Streamlit web application.

In [12]:
ml_model.fit(X.apply(clean_merchant_text), y)

joblib.dump({
    'model': ml_model,
    'merchant_dict': KNOWN_MERCHANT_DICT,
    'conf_threshold': 0.45,
    'accuracy': float(accuracy),
    'macro_f1': float(macro_f1),
    'unseen_accuracy': float(unseen_acc_hyb),
    'unseen_macro_f1': float(unseen_f1_hyb),
    'source': 'Combined Public + Local Hybrid Dataset' if not is_demo else 'Local Hybrid Training Dataset',
    'demo_data': bool(is_demo)
}, 'model.joblib')

print(f"Final model trained on all {len(X)} transactions and saved to model.joblib!")

Final model trained on all 11367 transactions and saved to model.joblib!


## Cell 12 - Try our Two-Stage Hybrid model manually

Let's test four different transactions:
1. `UPI/923847/ZOMATO` -> Famous brand caught by **Stage 1 (Dictionary)**.
2. `POS/89341/UBER` -> Famous brand caught by **Stage 1 (Dictionary)**.
3. `POS DEBIT 583920 MEDPLUS PHARMACY AND CHEMIST` -> Local pharmacy caught by **Stage 2 (Word + Char N-Gram ML)** with >90% confidence!
4. `UPI/P2M/818192/X99Z UNKNOWN VENDOR` -> Mysterious vendor that now gets a low confidence score (`< 0.45`) and triggers `[NEEDS REVIEW]`!

In [13]:
new_transactions = [
    'UPI/923847/ZOMATO',
    'POS/89341/UBER',
    'POS DEBIT 583920 MEDPLUS PHARMACY AND CHEMIST',
    'UPI/P2M/818192/X99Z UNKNOWN VENDOR'
]

manual_test_df = hybrid_predict_df(new_transactions, ml_model)
for _, row in manual_test_df.iterrows():
    review_tag = " [NEEDS REVIEW]" if row['needs_review'] else ""
    print(f"{row['description']}  ->  {row['predicted']} ({row['source']}, conf={row['confidence']}){review_tag}")
manual_test_df

UPI/923847/ZOMATO  ->  Food & Drinks (Stage 1: Dictionary, conf=1.0)
POS/89341/UBER  ->  Travel & Transport (Stage 1: Dictionary, conf=1.0)
POS DEBIT 583920 MEDPLUS PHARMACY AND CHEMIST  ->  Health & Fitness (Stage 2: ML (Word+Char TF-IDF), conf=0.946)
UPI/P2M/818192/X99Z UNKNOWN VENDOR  ->  Shopping (Stage 2: ML (Word+Char TF-IDF), conf=0.11) [NEEDS REVIEW]


,description,predicted,confidence,source,needs_review
0,UPI/923847/ZOMATO,Food & Drinks,1.000,Stage 1: Dictionary,False
1,POS/89341/UBER,Travel & Transport,1.000,Stage 1: Dictionary,False
2,POS DEBIT 583920 MEDPLUS PHARMACY AND CHEMIST,Health & Fitness,0.946,Stage 2: ML (Word+Char TF-IDF),False
3,UPI/P2M/818192/X99Z UNKNOWN VENDOR,Shopping,0.110,Stage 2: ML (Word+Char TF-IDF),True


## Cell 13 - Context-Aware Anomaly Detection with Isolation Forest

In a basic setup, running Isolation Forest on a single column (`Debit`) flags both the largest amount **and the smallest amount** (like a ₹95 bike ride or ₹180 tea!) because random tree splits isolate both ends of a 1D number line. Also, ₹5,000 is normal for `Investments`, and ₹2,000 is normal for `Shopping`, but ₹8,400 is unusually high for `Food & Drinks`.

**How we fixed it:**
1. **Two Features instead of 1D:** We feed Isolation Forest both `log_debit` ($\log(1 + \text{Debit})$) and `cat_deviation` (how far the transaction is above its own predicted category's median in log space).
2. **High-Spend Filter Only:** We only flag `unusual = True` when the transaction is **above** its category median (`cat_deviation > 0.5`), so cheap everyday purchases are never falsely flagged!

In [14]:
sample = pd.read_csv('data/sample_statement.csv')
pred_info = hybrid_predict_df(sample['Description'], ml_model)
sample['Category'] = pred_info['predicted'].values
sample['Method'] = pred_info['source'].values
sample['Confidence'] = pred_info['confidence'].values
sample['Needs_Review'] = pred_info['needs_review'].values

sample['Debit'] = pd.to_numeric(sample['Debit'], errors='coerce').fillna(0.0)
sample['Unusual'] = False
sample['Cat_Median'] = 0.0

debit_mask = sample['Debit'] > 0
if debit_mask.sum() >= 5:
    debits = sample.loc[debit_mask].copy()
    cat_medians = debits.groupby('Category')['Debit'].transform('median')
    sample.loc[debit_mask, 'Cat_Median'] = cat_medians
    
    log_amt = np.log1p(debits['Debit'])
    log_med = np.log1p(cat_medians)
    cat_dev = log_amt - log_med
    
    X_anom = pd.DataFrame({
        'log_debit': log_amt,
        'cat_deviation': cat_dev
    })
    
    anomaly_model = IsolationForest(contamination=0.12, random_state=42)
    raw_unusual = (anomaly_model.fit_predict(X_anom) == -1)
    
    # Only flag high spending spikes (strictly above the category median), never small debits!
    sample.loc[debit_mask, 'Unusual'] = (raw_unusual & (cat_dev > 0.5)).values

print("Context-Aware Anomaly Detection Results:")
sample[['Date', 'Description', 'Debit', 'Category', 'Cat_Median', 'Method', 'Unusual', 'Needs_Review']]

Context-Aware Anomaly Detection Results:


,Date,Description,Debit,Category,Cat_Median,Method,Unusual,Needs_Review
0,2026-11-01,NEFT CR-839201928374-SALARY CREDIT,0.0,Income,0.0,Stage 2: ML (Word+Char TF-IDF),False,False
1,2026-11-02,UPI/P2M/738291048291/ZOMATO,380.0,Food & Drinks,380.0,Stage 1: Dictionary,False,False
2,2026-11-03,UPI/P2M/948271625384/SWIGGY,420.0,Food & Drinks,380.0,Stage 1: Dictionary,False,False
3,2026-11-04,POS DEBIT 647382910293 SHARMA SWEETS AND RESTA...,290.0,Food & Drinks,380.0,Stage 2: ML (Word+Char TF-IDF),False,False
4,2026-11-05,UPI/P2M/583920192837/TEA JUNCTION CAFE,180.0,Food & Drinks,380.0,Stage 2: ML (Word+Char TF-IDF),False,False
5,2026-11-06,UPI/P2M/473829102938/ROYAL CAFE AND BAKERY,8400.0,Food & Drinks,380.0,Stage 2: ML (Word+Char TF-IDF),True,False
6,2026-11-07,UPI/P2M/382910293847/BLINKIT,650.0,Groceries,815.0,Stage 1: Dictionary,False,False
7,2026-11-08,UPI/P2M/291029384756/ZEPTO,540.0,Groceries,815.0,Stage 1: Dictionary,False,False
8,2026-11-09,POS DEBIT 192837465019 KRISHNA SUPERMARKET,1200.0,Groceries,815.0,Stage 2: ML (Word+Char TF-IDF),False,False
9,2026-11-10,UPI/P2M/918273645019/BIGBASKET,980.0,Groceries,815.0,Stage 1: Dictionary,False,False
